In [0]:
import sys
# Ensure parent directory is in path for module resolution
sys.path.append("/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline/")
from utils.metrics_query import MetricsQuery

mq = MetricsQuery(spark)

In [0]:
import sys
import os
from datetime import datetime
from pyspark.sql.functions import col, date_format

# --------------------------------------------------
# 1. Widgets & Setup
# --------------------------------------------------
dbutils.widgets.text("customer_id", "")
dbutils.widgets.text("object_name", "")
dbutils.widgets.text("source_system", "salesforce")
dbutils.widgets.text("sf_catalog_table", "")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
sf_catalog_table = dbutils.widgets.get("sf_catalog_table")
load_type = "incremental"

# --------------------------------------------------
# 2. Import Configuration dynamically
# --------------------------------------------------
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
abs_project_root = os.path.abspath(project_root)

if abs_project_root not in sys.path:
    sys.path.append(abs_project_root)

from utils.config import DataLakeConfig

# Instantiate the config
config = DataLakeConfig(source_system, customer_id, object_name, load_type)

# Optimize shuffle for small clusters
spark.conf.set("spark.sql.shuffle.partitions", 4)

# Generate Timestamped path via Config Helper
now = datetime.now()
incremental_path = config.get_landing_zone_timestamped_path(source_system, customer_id, object_name, load_type, now)

print(f"🚀 Starting Incremental Load to: {incremental_path}")

# --------------------------------------------------
# 3. Get the last watermark
# --------------------------------------------------
try:
    watermark_df = spark.sql(f"""
    SELECT last_processed_timestamp
    FROM ingestion_metadata.watermarks
    WHERE source_system='{source_system}'
    AND object_name='{object_name}'
    """)
    rows = watermark_df.collect()
    if len(rows) == 0:
        raise Exception("Watermark missing.")
    watermark = rows[0][0]
except Exception as e:
    raise Exception(f"❌ Watermark not initialized. Please run historical load first. Error: {str(e)}")

print(f"📍 Last Watermark: {watermark}")

# 4. Pre-Fetch: Query Salesforce FIRST to get the NEW max timestamp
new_ts_df = spark.sql(f"""
SELECT MAX(SystemModstamp) as max_ts
FROM {sf_catalog_table}
WHERE SystemModstamp > TIMESTAMP('{watermark}')
""")

new_ts = new_ts_df.first()[0]

if new_ts is None:
    print("✅ No new records found in Salesforce. Exiting gracefully.")
    dbutils.notebook.exit("0")

print(f"🎯 New records found up to: {new_ts}")

# 5. Pull the data using BOTH bounds
df_incremental = spark.sql(f"""
SELECT *
FROM {sf_catalog_table}
WHERE SystemModstamp > TIMESTAMP('{watermark}')
  AND SystemModstamp <= TIMESTAMP('{new_ts}')
""")

# Format the Timestamp to exact String format 
df_formatted = df_incremental.withColumn("SystemModstamp", date_format(col("SystemModstamp"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))

try:
    # 6. Write to S3 using config path
    (
        df_formatted
        .repartition(4)
        .write
        .mode("append")
        .format("parquet")
        .option("compression", "snappy")
        .save(incremental_path)
    )
    print("💾 Write to S3 completed successfully.")
    
except Exception as e:
    print(f"❌ Failed during write operation: {str(e)}")
    raise e

# 7. Update Watermark with 1-Minute Lookback
print("Updating watermark table safely using MERGE...")

spark.sql(f"""
    MERGE INTO ingestion_metadata.watermarks AS target
    USING (
        SELECT '{source_system}' AS source_system, 
               '{object_name}' AS object_name, 
               TIMESTAMP('{new_ts}') - INTERVAL 1 MINUTE AS last_processed_timestamp
    ) AS source
    ON target.source_system = source.source_system AND target.object_name = source.object_name
    WHEN MATCHED THEN
        UPDATE SET target.last_processed_timestamp = source.last_processed_timestamp
    WHEN NOT MATCHED THEN
        INSERT (source_system, object_name, last_processed_timestamp)
        VALUES (source.source_system, source.object_name, source.last_processed_timestamp)
""")

print(f"✅ Watermark updated safely (minus 1 minute).")


In [0]:
# Cell 8 — use actual resolved table name
# No table is created, so skip row count and just save stats with rows_processed=None
mq.save_stats(days=30, rows_processed=None)
print(f":white_check_mark: Metrics saved | rows_processed: None for run_id: {mq._run_id}")


In [0]:
# # =============================================================================
# # Notebook: 02_Incremental_Load_SF_to_S3 (Optimized — Lakehouse Federation)
# # Strategy : Push both-sided filter to federation connector so Salesforce
# #            only returns rows in the watermark window.
# #            One scan only — we cache and derive new_ts from cache, no
# #            second federation round-trip.
# # Runs     : Every 15 minutes.
# # Compute  : Both Serverless and Job Cluster.
# # =============================================================================

# # --------------------------------------------------------------------------
# # 0. Performance Configuration (tuned for small SF incremental batches)
# # --------------------------------------------------------------------------
# spark.conf.set("spark.sql.adaptive.enabled", "true")
# spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
# spark.conf.set("spark.sql.adaptive.advisoryPartitionSizeInBytes", "67108864")  # 64 MB

# # SF incremental is tiny — 2-8 shuffle partitions is enough
# spark.conf.set("spark.sql.shuffle.partitions", "4")

# # S3 write
# spark.conf.set("spark.hadoop.fs.s3a.fast.upload", "true")
# spark.conf.set("spark.hadoop.fs.s3a.multipart.size", "67108864")
# spark.conf.set("spark.hadoop.fs.s3a.threads.max", "8")
# spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")

# spark.conf.set("spark.sql.parquet.compression.codec", "snappy")
# spark.conf.set("spark.sql.parquet.mergeSchema", "false")

# # --------------------------------------------------------------------------
# # 1. Widgets
# # --------------------------------------------------------------------------
# dbutils.widgets.text("customer_id",      "")
# dbutils.widgets.text("object_name",      "")
# dbutils.widgets.text("source_system",    "salesforce")
# dbutils.widgets.text("bucket_path",      "")
# dbutils.widgets.text("sf_catalog_table", "")

# customer_id      = dbutils.widgets.get("customer_id")
# object_name      = dbutils.widgets.get("object_name")
# source_system    = dbutils.widgets.get("source_system")
# bucket_path      = dbutils.widgets.get("bucket_path").rstrip("/")
# sf_catalog_table = dbutils.widgets.get("sf_catalog_table")

# if not all([customer_id, object_name, bucket_path, sf_catalog_table]):
#     raise Exception("❌ Missing required widget parameters.")

# from datetime import datetime, timedelta
# from pyspark.sql import functions as F

# now  = datetime.utcnow()
# yyyy, mm, dd, hh = now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H")
# minute_bucket = (now.minute // 15) * 15
# incremental_path = (
#     f"{bucket_path}/landing-zone/{source_system}/{customer_id}/{object_name}"
#     f"/incremental/{yyyy}/{mm}/{dd}/{hh}/{minute_bucket:02d}"
# )

# print(f"=== Salesforce Incremental Load ===")
# print(f"Customer  : {customer_id}")
# print(f"Object    : {object_name}")
# print(f"S3 Path   : {incremental_path}")

# # --------------------------------------------------------------------------
# # 2. Read watermark
# # --------------------------------------------------------------------------
# wm_rows = spark.sql(f"""
#     SELECT last_processed_timestamp
#     FROM   ingestion_metadata.watermarks
#     WHERE  source_system = '{source_system}'
#       AND  object_name   = '{object_name}'
# """).collect()

# if not wm_rows:
#     raise Exception(
#         f"❌ Watermark not found for {source_system}/{object_name}. "
#         "Run the historical load notebook first."
#     )

# watermark = wm_rows[0][0]
# print(f"Watermark : {watermark}")

# # --------------------------------------------------------------------------
# # 3. Load incremental data — ONE scan with both-sided bounds
# #
# #    Upper bound = now (pipeline time), not MAX() from SF.
# #    This avoids a second federation query for MAX and prevents
# #    future-dated records from polluting the watermark.
# #
# #    The filter is pushed down to the Salesforce connector at the
# #    federation layer — only qualifying rows are fetched over the wire.
# # --------------------------------------------------------------------------
# lower_bound = watermark.strftime("%Y-%m-%d %H:%M:%S") if hasattr(watermark, 'strftime') else str(watermark)
# upper_bound = now.strftime("%Y-%m-%d %H:%M:%S")

# df_incremental = spark.sql(f"""
#     SELECT *
#     FROM   {sf_catalog_table}
#     WHERE  SystemModstamp >  TIMESTAMP('{lower_bound}')
#       AND  SystemModstamp <= TIMESTAMP('{upper_bound}')
# """).cache()

# # --------------------------------------------------------------------------
# # 4. Early exit if no new data (count on cache is instant after first fetch)
# # --------------------------------------------------------------------------
# record_count = df_incremental.count()

# if record_count == 0:
#     print(f"✅ No new records since {watermark}. Exiting.")
#     df_incremental.unpersist()
#     dbutils.notebook.exit("0")

# print(f"New records: {record_count:,}")

# # Derive new watermark from cached data — no second SF round-trip
# new_ts = df_incremental.select(F.max("SystemModstamp")).collect()[0][0]
# print(f"New max ts : {new_ts}")

# # --------------------------------------------------------------------------
# # 5. Write to S3
# #    SF incremental is tiny. coalesce(1) avoids too many small files.
# #    If object_name is Account/Contact/Opportunity (larger), increase limit.
# # --------------------------------------------------------------------------
# output_partitions = max(1, record_count // 10_000)   # ~10k rows per file; SF is small

# (
#     df_incremental
#     .coalesce(output_partitions)
#     .write
#     .mode("append")
#     .option("compression", "snappy")
#     .format("parquet")
#     .save(incremental_path)
# )

# df_incremental.unpersist()
# print(f"✅ Write complete → {incremental_path}  ({output_partitions} file(s))")

# # --------------------------------------------------------------------------
# # 6. Update watermark atomically with MERGE
# # --------------------------------------------------------------------------
# watermark_ts = new_ts - timedelta(minutes=1)

# spark.sql(f"""
#     MERGE INTO ingestion_metadata.watermarks AS tgt
#     USING (
#         SELECT
#             '{source_system}' AS source_system,
#             '{object_name}'   AS object_name,
#             TIMESTAMP('{watermark_ts}') AS last_processed_timestamp
#     ) AS src
#     ON  tgt.source_system = src.source_system
#     AND tgt.object_name   = src.object_name
#     WHEN MATCHED THEN UPDATE SET
#         tgt.last_processed_timestamp = src.last_processed_timestamp
#     WHEN NOT MATCHED THEN INSERT *
# """)

# print(f"✅ Watermark updated to {watermark_ts}  (new_ts minus 1 min).")
# dbutils.notebook.exit("Success")
